# 07 · Metals — geometries, isomers, chelates, TS, binding modes

A metal centre becomes a UFF-typeable **carbon surrogate** (bonds stripped), held by distance + angle constraints for the chosen polyhedron, then restored — plain RDKit, no xtb. `rx.metal(smiles, geometry)` enumerates the distinct **coordination isomers** as an `IsomerSet` to pick from.

**Rendering note.** xyzrender detects ordinary bonds (including metal–donor coordination) from the geometry on its own — we only name the *special* ones: **non-covalent contacts** (H-bonds, chalcogen, cation-π) via `nci_bonds=[(i, j)]` (dotted), and **genuine forming/breaking TS bonds** via `ts_bonds=[(i, j)]` (dashed). Distances/angles are labelled with `labels=["i j d"]` / `"i j k a"`. xyzrender is 1-indexed, so atom indices are passed `+1`.

In [ ]:
import tempfile
import numpy as np
import rxembed as rx
import xyzrender
from rdkit import RDLogger
from rdkit.Chem import rdMolTransforms as T
from rxembed.constraints.metal import metal_indices
from rxembed.embed.dispatch import _xyz_to_mol

RDLogger.DisableLog("rdApp.*")
rx.set_verbose("INFO")


def xyz(ens):
    return ens.lowest(1).dump(tempfile.mktemp(suffix=".xyz"))  # 1 conformer -> a path xyzrender reads


def mxyz(ens):
    return ens.dump(tempfile.mktemp(suffix=".xyz"))

## Coordination isomers — the permutations

Square-planar MA₂B₂ → **cis / trans**; octahedral MA₃B₃ → **mer / fac**. `.summary()` tags each with its **per-vertex arrangement** and Λ/Δ **chirality** — the unambiguous, name-agnostic identity. Pick one with `.select(index=…)`, `.select(arrangement=…)`, or `.select(chirality=…)` (the coarse cis/trans/mer/fac name is intentionally not the selector — it's ambiguous once several isomers share it).

In [ ]:
print("square-planar Pd (cis/trans):")
rx.metal("CCCN[Pd](Cl)(Cl)NCCC", "square_planar").summary()
print("\noctahedral Co(NH3)3Cl3 (mer/fac):")
rx.metal("N[Co](N)(N)(Cl)(Cl)Cl", "octahedral").summary()

The **fac** Co isomer: the coordination sphere drawn as dotted dative bonds (`nci_bonds`) makes the 3 N / 3 Cl faces obvious; element labels via `idx='s'`.

In [ ]:
co = rx.metal("N[Co](N)(N)(Cl)(Cl)Cl", "octahedral").select(index=1)  # the fac isomer (index 1 in .summary())
ens = rx.embed(co).mc(preset="ensemble").prune()
p = xyz(ens)
xyzrender.render(p, no_hy=True, idx="s")

The **trans** Pd isomer — the two Cl sit ~180° across the metal (labelled Cl–Pd–Cl angle).

In [ ]:
trans = rx.metal("CCCN[Pd](Cl)(Cl)NCCC", "square_planar").select(index=1)  # the trans isomer (index 1)
ens = rx.embed(trans).mc(preset="ensemble").prune()
cl = [d for d in trans.donors if ens.mol.GetAtomWithIdx(d).GetSymbol() == "Cl"]
print(f"{len(ens)} conformers | Cl-Pd-Cl = {ens.measure((cl[0], trans.metal, cl[1]))['mean']:.0f}° (trans)")
p = xyz(ens)
xyzrender.render(
    p, no_hy=True, labels=[f"{cl[0] + 1} {trans.metal + 1} {cl[1] + 1} a"]
)  # label the trans Cl-Pd-Cl angle

## Across coordination geometries

One surface spans the polyhedra — linear, tetrahedral, square-planar, octahedral (and trigonal-bipyramidal / square-pyramidal for 5-coordinate).

In [ ]:
for g, smi in {
    "linear": "[Cl][Au][Cl]",
    "tetrahedral": "Cl[Zn](Cl)(Cl)Cl",
    "square_planar": "CCCN[Pd](Cl)(Cl)NCCC",
    "octahedral": "N[Co](N)(N)(Cl)(Cl)Cl",
}.items():
    isos = rx.metal(smi, g)
    print(f"  {g:14s}: {len(isos)} isomer(s) {[i.label for i in isos]}")

## Metal-centre chirality — Λ / Δ, and the robust conformer pipeline

A bis/tris-chelate metal can be a genuine stereocentre with **no organic stereocentre in sight** — the handedness of the coordination sphere itself. rxembed tags it Λ/Δ (a point-group-parity descriptor), so `[Co(en)₂Cl₂]` enumerates the textbook three: **trans** (achiral) and the enantiomeric **cis-Δ / cis-Λ**. Select on `chirality=`. Then run the full metal conformer pipeline — **relax the seeds → MC search → re-relax → dedup** (`embed(n).minimize().mc().minimize().prune()`) — pull the distinct modes with `representatives()`, and grab one by position with `reps[i]`.

In [ ]:
isos = rx.metal("Cl[Co]12(Cl)(NCCN1)NCCN2", "octahedral")  # [Co(en)2Cl2] — bis-chelate + 2 Cl
isos.summary()  # trans | achiral ;  cis | Δ ;  cis | Λ
delta = isos.select(chirality="Δ")  # pick the Δ enantiomer by its handedness (name-agnostic, unambiguous)
ens = rx.embed(delta, n=8).minimize().mc(preset="ensemble").minimize().prune()  # relax -> search -> re-relax -> dedup
reps = ens.representatives()
print(f"cis-Δ [Co(en)2Cl2]: {ens.n} conformers -> {reps.n} distinct mode(s)")
xyzrender.render(reps[0].dump(tempfile.mktemp(suffix=".xyz")), no_hy=True, idx="s")  # reps[0] — the new [i] indexing

## Chelating ligands — bidentate

A chelate's donors sit on one ligand fragment; rxembed detects same-fragment donors and widens their L–M–L window (a rigid backbone bite is distorted off the ideal) while still separating cis from trans. Ethylenediamine on Pd (a 5-membered chelate ring, N–Pd–N bite ~82°) + two chloride:

In [ ]:
en = rx.metal("Br[Pd]1(Cl)NCCN1", "square_planar")  # the NCCN ring pins both N to Pd
iso = en.select()
ens = rx.embed(iso, n=10).minimize().mc(preset="ensemble").minimize().prune()
nn = [d for d in iso.donors if ens.mol.GetAtomWithIdx(d).GetSymbol() == "N"]
print(f"{len(ens)} conformers | en bite N-Pd-N = {ens.measure((nn[0], iso.metal, nn[1]))['mean']:.0f}°")
xyzrender.render(
    xyzrender.load(
        ens.dump(tempfile.mktemp(suffix=".xyz")), ensemble=len(ens) > 1, ensemble_color="spectral", nci_detect=True
    ),
    no_hy=True,
)

## Side-on η² coordination

When a ligand's two coordinating atoms are **directly π-bonded** (an alkene, alkyne, imine, phosphaalkene), it binds *side-on* — both atoms grip through the π bond. rxembed detects the multiple-bonded donor pair and **holds that bond at its natural length**, so the two metal pulls can't stretch the C=C / C≡C apart (which would otherwise tear it and blow the coordination sphere out). Here a Ni acyl-malonate with a side-on C=C: both Ni–C distances bind and the double bond stays intact.

In [ ]:
iso = rx.metal(
    "COC(=O)[C]12->[Ni+2]3(<-[O-]C(=O)C(c4ccccc4)[N-]->3c3ccccc3)<-[C]=1(C(=O)OC)C2(C)C(C)(C)C",
    "square_planar",
).select(index=0)
pi = next(  # the directly π-bonded donor pair — the side-on C=C
    (a, b)
    for a in iso.donors
    for b in iso.donors
    if a < b and (bd := iso.mol.GetBondBetweenAtoms(a, b)) and bd.GetBondTypeAsDouble() >= 2
)
ens = rx.embed(iso, n=8).minimize().mc(preset="ensemble").minimize().prune()
c = ens.mol.GetConformer(ens.ids[0])
print(
    f"side-on C=C {pi}: {ens.n} confs | Ni-C {T.GetBondLength(c, iso.metal, pi[0]):.2f}/"
    f"{T.GetBondLength(c, iso.metal, pi[1]):.2f} Å (both bind) | C=C {T.GetBondLength(c, *pi):.2f} Å (held intact)"
)
xyzrender.render(xyz(ens), no_hy=True, labels=[f"{iso.metal + 1} {pi[0] + 1} d", f"{iso.metal + 1} {pi[1] + 1} d"])

## Vacant sites — a coordination pocket

Name a geometry with **more vertices than donors** and the empty vertex is left as a pocket (`·` in the arrangement) for a substrate to bind later. A 3-donor Pd in a square-planar field leaves one site open.

In [ ]:
pocket = rx.metal("CCN[Pd](Cl)Cl", "square_planar")  # N + 2 Cl = 3 donors, 4 vertices
pocket.summary()  # '·' marks the vacant coordination site
iso = pocket.select(index=0)  # the cis pocket (both Cl cis; the '·' vacancy sits opposite the lone Cl)
ens = rx.embed(iso).mc(preset="ensemble").prune()
xyzrender.render(xyz(ens), no_hy=True, idx="s")  # the '·' vacancy sits opposite the lone Cl

## A real metal TS — reacting core frozen, spectator ferrocene held

A **bimetallic** Mn/Fe transition state: H₂ activation at Mn (H–H breaking, Mn–H / substrate N–H, O–H forming), with a *spectator* ferrocene. Freeze the reacting core — grafted back to **0.000 Å** on every pose while the periphery is searched; both metals restored. Here `ts_bonds=` is genuinely correct (real forming/breaking bonds), labelled with their live lengths — note the ~0.86 Å H–H being cleaved.

In [ ]:
mn = "structures/mn-h2.xyz"
rc = [1, 5, 63, 64, 65, 66]  # reacting-core atoms (from a graphrc trajectory in practice)
bonds = [(5, 65), (64, 66), (1, 64), (65, 66), (63, 64)]  # forming / breaking bonds held during embed
Pin = _xyz_to_mol(mn, 0).GetConformer().GetPositions()
print("metals:", [(i, _xyz_to_mol(mn, 0).GetAtomWithIdx(i).GetSymbol()) for i in metal_indices(_xyz_to_mol(mn, 0))])
ens = rx.embed(mn, fix=rc, n=6).mc(preset="ensemble").prune()
drift = max(
    abs(T.GetBondLength(ens.mol.GetConformer(c), i, j) - float(np.linalg.norm(Pin[i] - Pin[j])))
    for c in ens.ids
    for i, j in bonds
)
print(
    f"{len(ens)} conformers | reacting core held to {drift:.4f} Å | "
    f"metals: {[ens.mol.GetAtomWithIdx(i).GetSymbol() for i in metal_indices(ens.mol)]}"
)
xyzrender.render(
    xyzrender.load(mxyz(ens), ensemble=len(ens) > 1, ensemble_color="spectral"),  # guard: ensemble needs >=2 frames
    no_hy=True,
    ts_bonds=[(i + 1, j + 1) for i, j in bonds],
    labels=[f"{i + 1} {j + 1} d" for i, j in bonds],
)

With the reacting core frozen, the **free** coordination sites around Mn are enumerated (mer/fac of the spectators). One arrangement (`fac1`) is geometrically infeasible and embeds to 0 conformers — honestly reported, not hidden — while **mer** embeds cleanly:

In [ ]:
isos = rx.metal(mn, "octahedral", center="Mn", fix=rc)
for iso in isos:
    e = rx.embed(iso, n=2).mc(preset="rapid").prune()
    d = max(
        (
            abs(T.GetBondLength(e.mol.GetConformer(c), i, j) - float(np.linalg.norm(Pin[i] - Pin[j])))
            for c in e.ids
            for i, j in bonds
        ),
        default=0.0,
    )
    print(f"  {iso.label:6s}: {len(e):2d} confs | core held {d:.4f} Å")
mer = rx.embed(isos[3], n=3).mc(preset="rapid").prune()
xyzrender.render(xyz(mer), no_hy=True, ts_bonds=[(i + 1, j + 1) for i, j in bonds])  # the mer isomer

The same frozen-TS machinery spans very different real organometallic TS (core held to 0.000 Å, every metal restored): **mn-h2** (H₂ activation, 78 atoms), **mn-hy** (a larger Mn hydride, 95 atoms), **ru-co** (a Ru carbonyl TS, 72 atoms).

In [ ]:
META = {  # reacting core + forming/breaking bonds per TS (from graphrc trajectories in practice)
    "mn-h2": {"rc": [1, 5, 63, 64, 65, 66], "bonds": [(5, 65), (64, 66), (1, 64), (65, 66), (63, 64)], "natoms": 78},
    "mn-hy": {"rc": [1, 67, 77], "bonds": [(67, 77), (1, 67)], "natoms": 95},
    "ru-co": {"rc": [0, 3, 63, 64, 66], "bonds": [(0, 64), (64, 66), (0, 63), (3, 66)], "natoms": 72},
}
for name in ["mn-h2", "mn-hy", "ru-co"]:
    rc2 = META[name]["rc"]
    b2 = META[name]["bonds"]
    pin = _xyz_to_mol(f"structures/{name}.xyz", 0).GetConformer().GetPositions()
    e = rx.embed(f"structures/{name}.xyz", fix=rc2, n=2).mc(preset="rapid").prune()
    drift = max(
        abs(T.GetBondLength(e.mol.GetConformer(c), i, j) - float(np.linalg.norm(pin[i] - pin[j])))
        for c in e.ids
        for i, j in b2
    )
    metals = [e.mol.GetAtomWithIdx(i).GetSymbol() for i in metal_indices(e.mol)]
    print(f"  {name:6s}: {META[name]['natoms']:3d} atoms | {len(e):2d} confs | core held {drift:.4f} Å | {metals}")

## Seating a substrate — and switching its binding mode

Reach a substrate donor into the vacant site with `coordinate=`. A Pt(amine)Cl₂ has one open site; seat an acetone carbonyl O there (a coordinative Pt–O bond, drawn dotted + labelled).

In [ ]:
isos = rx.metal("N[Pt](Cl)Cl.CC(C)=O", "square_planar")  # 3 donors + acetone -> one vacant site
iso = isos.select(index=0)  # the cis isomer (both Cl cis)
O = next(a.GetIdx() for a in iso.mol.GetAtoms() if a.GetSymbol() == "O")
seated = rx.embed(iso, coordinate=O, n=3).mc(preset="ensemble").prune()
print(f"acetone O seated: Pt-O = {seated.measure((iso.metal, O))['mean']:.2f} Å  ({len(seated)} confs)")
xyzrender.render(xyz(seated), no_hy=True, labels=[f"{iso.metal + 1} {O + 1} d"])  # label the new Pt-O coordinative bond

**Switching the binding mode:** the same substrate can grip *bifunctionally* — the O coordinated to Pt **and** an amine N–H···O hydrogen bond held at the same time. Two contacts, one pose (both dotted, both labelled).

In [ ]:
N = next(d for d in iso.donors if iso.mol.GetAtomWithIdx(d).GetSymbol() == "N")
H = next(n.GetIdx() for n in iso.mol.GetAtomWithIdx(N).GetNeighbors() if n.GetAtomicNum() == 1)
grip = rx.Contact(distances={(min(H, O), max(H, O)): (1.6, 2.2)}, angles={(N, H, O): (140.0, 180.0)}, label="NH..O")
bi = rx.embed(iso, coordinate=O, contacts=grip, n=3).mc(preset="rapid").prune()
print(
    f"bifunctional: Pt-O {bi.measure((iso.metal, O))['mean']:.2f} Å + H-O {bi.measure((H, O))['mean']:.2f} Å, "
    f"N-H-O {bi.measure((N, H, O))['mean']:.0f}°"
)
xyzrender.render(
    xyz(bi),
    no_hy=True,
    nci_bonds=[(H + 1, O + 1)],  # only the H...O is an NCI (dotted); Pt-O is a normal bond
    labels=[f"{iso.metal + 1} {O + 1} d", f"{H + 1} {O + 1} d"],
)